# eFootball YOLO (6-Class UI) — Colab 免费 T4 GPU 一键训练

- 一键打开（无需配置 GitHub Token，无需挂载 Google Drive）：
  https://colab.research.google.com/github/idhuanxiang/efootball-bot/blob/main/training/colab_train.ipynb
- 说明：Colab 分配免费 GPU 仍需登录一个 Google 账号；本笔记本不挂载 Drive、不写回仓库。
- 建议在菜单 `代码执行程序 -> 更改运行时类型` 中选择 **T4 GPU**，399+ 张图跑 80 epoch 约 **3~5 分钟**。

In [ ]:
!nvidia-smi
import pathlib, shutil, zipfile

REPO_DIR = pathlib.Path('/content/efootball-bot')
if REPO_DIR.exists():
    shutil.rmtree(REPO_DIR)
!git clone --depth 1 https://github.com/idhuanxiang/efootball-bot.git /content/efootball-bot
%cd /content/efootball-bot
!pip -q install -U ultralytics onnx onnxruntime pillow

ds_zip = pathlib.Path('training/dataset.zip')
if ds_zip.is_file() and not (pathlib.Path('training/dataset/images/train')).is_dir():
    with zipfile.ZipFile(ds_zip) as zf:
        zf.extractall('training/dataset')
    print('Unpacked training/dataset.zip')

In [ ]:
# 训练参数（增量续训可把 WEIGHTS 改为上次的 yolo26n_best.pt，FREEZE=10, EPOCHS=30）
EPOCHS = 80
IMGSZ = 640
BATCH = 16
FREEZE = 0
WEIGHTS = 'yolo26n.pt'
ENABLE_UI_AUG = True
SEED = 0

!python training/prepare_dataset.py
if ENABLE_UI_AUG:
    !python training/augment_ui.py --dataset training/dataset --ratio 0.35 --seed {SEED}

In [ ]:
!python training/train_yolo.py \
  --weights {WEIGHTS} \
  --epochs {EPOCHS} \
  --imgsz {IMGSZ} \
  --batch {BATCH} \
  --freeze {FREEZE} \
  --seed {SEED} \
  --device 0 \
  --workers 2

In [ ]:
!python training/verify_model.py \
  --frames training/negatives/images \
  --compare src/EfootballBot/models/yolo/yolo26n.onnx | tee training/out/verify_report.txt

In [ ]:
import pathlib, zipfile
from google.colab import files

bundle = pathlib.Path('training/out/yolo26n_colab_bundle.zip')
with zipfile.ZipFile(bundle, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in ['training/out/yolo26n_candidate.onnx', 'training/out/yolo26n_best.pt', 'training/out/verify_report.txt']:
        fp = pathlib.Path(p)
        if fp.is_file():
            zf.write(fp, fp.name)
print('Ready to download:', bundle)
files.download(str(bundle))